In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
CONTRASTIVE_EPOCHS = 3
CLASSIFIER_EPOCHS = 10
CONTRASTIVE_LR = 2e-5
CLASSIFIER_LR = 1e-3
MARGIN = 0.3

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

HARD_NEGATIVE_MAP = {
    "Clear Reply": "Ambivalent",
    "Ambivalent": "Clear Reply",
    "Clear Non-Reply": "Ambivalent"
}

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_full_df['clarity_label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")
print(f"\nClass distribution (train):")
print(train_df['clarity_label'].value_counts())

In [ ]:
class TripletDataset(Dataset):
    """
    Dataset that returns triplets: (anchor, positive, negative)
    - Anchor: A random sample
    - Positive: Another sample from the SAME class as anchor
    - Negative: A sample from a DIFFERENT class, preferentially from the "hard" class
    """
    
    def __init__(self, df, tokenizer, max_length, hard_negative_map, hard_negative_prob=0.8):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.hard_negative_map = hard_negative_map
        self.hard_negative_prob = hard_negative_prob
        
        self.label_to_indices = {}
        for label in label2id.keys():
            self.label_to_indices[label] = self.df[self.df['clarity_label'] == label].index.tolist()
    
    def __len__(self):
        return len(self.df)
    
    def _format_text(self, row):
        return f"Question: {row['question']}\nAnswer: {row['interview_answer']}"
    
    def _tokenize(self, text):
        return self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
    
    def __getitem__(self, idx):
        anchor_row = self.df.iloc[idx]
        anchor_label = anchor_row['clarity_label']
        anchor_text = self._format_text(anchor_row)
        
        positive_indices = [i for i in self.label_to_indices[anchor_label] if i != idx]
        if len(positive_indices) == 0:
            positive_indices = [idx]
        positive_idx = random.choice(positive_indices)
        positive_row = self.df.iloc[positive_idx]
        positive_text = self._format_text(positive_row)
        
        if random.random() < self.hard_negative_prob:
            hard_negative_label = self.hard_negative_map[anchor_label]
            negative_indices = self.label_to_indices[hard_negative_label]
        else:
            all_other_labels = [l for l in label2id.keys() if l != anchor_label]
            random_other_label = random.choice(all_other_labels)
            negative_indices = self.label_to_indices[random_other_label]
        
        negative_idx = random.choice(negative_indices)
        negative_row = self.df.iloc[negative_idx]
        negative_text = self._format_text(negative_row)
        
        anchor_enc = self._tokenize(anchor_text)
        positive_enc = self._tokenize(positive_text)
        negative_enc = self._tokenize(negative_text)
        
        return {
            'anchor_input_ids': anchor_enc['input_ids'].squeeze(0),
            'anchor_attention_mask': anchor_enc['attention_mask'].squeeze(0),
            'positive_input_ids': positive_enc['input_ids'].squeeze(0),
            'positive_attention_mask': positive_enc['attention_mask'].squeeze(0),
            'negative_input_ids': negative_enc['input_ids'].squeeze(0),
            'negative_attention_mask': negative_enc['attention_mask'].squeeze(0),
            'anchor_label': label2id[anchor_label]
        }

In [ ]:
class ClassificationDataset(Dataset):
    """Simple dataset for classification stage"""
    
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"Question: {row['question']}\nAnswer: {row['interview_answer']}"
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': label2id[row['clarity_label']]
        }

In [ ]:
class RobertaContrastiveEncoder(nn.Module):
    """
    RoBERTa encoder with mean pooling for contrastive learning.
    Outputs a fixed-size embedding vector for each input sequence.
    """
    
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size
    
    def mean_pooling(self, last_hidden_state, attention_mask):
        """
        Mean pooling over token embeddings, weighted by attention mask.
        
        Args:
            last_hidden_state: [batch_size, seq_len, hidden_size]
            attention_mask: [batch_size, seq_len]
        
        Returns:
            pooled: [batch_size, hidden_size]
        """
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask
    
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        embeddings = self.mean_pooling(last_hidden_state, attention_mask)
        # Removed F.normalize - using unnormalized embeddings to prevent collapse
        return embeddings
    
    def freeze_encoder(self):
        for param in self.roberta.parameters():
            param.requires_grad = False
    
    def unfreeze_encoder(self):
        for param in self.roberta.parameters():
            param.requires_grad = True

In [ ]:
class LinearClassifier(nn.Module):
    """Simple linear classifier on top of frozen embeddings"""
    
    def __init__(self, input_dim, num_classes, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(input_dim, num_classes)
    
    def forward(self, embeddings):
        x = self.dropout(embeddings)
        return self.classifier(x)

In [ ]:
train_triplet_dataset = TripletDataset(
    train_df, tokenizer, MAX_LENGTH, HARD_NEGATIVE_MAP, hard_negative_prob=0.8
)
train_triplet_loader = DataLoader(
    train_triplet_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

print(f"Training triplets: {len(train_triplet_dataset)}")
print(f"Batches per epoch: {len(train_triplet_loader)}")

In [ ]:
encoder = RobertaContrastiveEncoder(MODEL_NAME).to(device)
encoder.roberta.gradient_checkpointing_enable()
triplet_loss_fn = nn.TripletMarginLoss(margin=MARGIN, p=2)

optimizer = torch.optim.AdamW(encoder.parameters(), lr=CONTRASTIVE_LR, weight_decay=0.01)
total_steps = (len(train_triplet_loader) // GRADIENT_ACCUMULATION_STEPS) * CONTRASTIVE_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Encoder hidden size: {encoder.hidden_size}")
print(f"Total training steps: {total_steps}")

In [ ]:
print("Stage 1: Contrastive Learning with Triplet Loss")
print("=" * 50)

encoder.train()
for epoch in range(CONTRASTIVE_EPOCHS):
    epoch_loss = 0.0
    progress_bar = tqdm(train_triplet_loader, desc=f"Epoch {epoch+1}/{CONTRASTIVE_EPOCHS}")
    
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(progress_bar):
        anchor_ids = batch['anchor_input_ids'].to(device)
        anchor_mask = batch['anchor_attention_mask'].to(device)
        positive_ids = batch['positive_input_ids'].to(device)
        positive_mask = batch['positive_attention_mask'].to(device)
        negative_ids = batch['negative_input_ids'].to(device)
        negative_mask = batch['negative_attention_mask'].to(device)
        
        anchor_emb = encoder(anchor_ids, anchor_mask)
        positive_emb = encoder(positive_ids, positive_mask)
        negative_emb = encoder(negative_ids, negative_mask)
        
        loss = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        
        # Debug: Monitor distances every 50 steps
        if batch_idx % 50 == 0:
            with torch.no_grad():
                dist_pos = F.pairwise_distance(anchor_emb, positive_emb, p=2).mean().item()
                dist_neg = F.pairwise_distance(anchor_emb, negative_emb, p=2).mean().item()
                print(f"\n[Step {batch_idx}] Dist_Pos: {dist_pos:.4f}, Dist_Neg: {dist_neg:.4f}, Separation: {dist_neg - dist_pos:.4f}")
        
        loss.backward()
        
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        progress_bar.set_postfix({'loss': f'{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}'})    print(f"Epoch {epoch+1} - Average Triplet Loss: {avg_loss:.4f}")

    

    print("\nContrastive training complete!")
    avg_loss = epoch_loss / len(train_triplet_loader)

In [ ]:
print("Freezing encoder weights...")
encoder.freeze_encoder()
encoder.eval()

trainable_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in encoder.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

In [ ]:
def extract_embeddings(encoder, dataloader, device):
    """Extract embeddings for all samples in a dataloader"""
    encoder.eval()
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting embeddings"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']
            
            embeddings = encoder(input_ids, attention_mask)
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)
    
    return torch.cat(all_embeddings, dim=0), torch.cat(all_labels, dim=0)

train_cls_dataset = ClassificationDataset(train_df, tokenizer, MAX_LENGTH)
val_cls_dataset = ClassificationDataset(val_df, tokenizer, MAX_LENGTH)
test_cls_dataset = ClassificationDataset(test_df, tokenizer, MAX_LENGTH)

train_cls_loader = DataLoader(train_cls_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_cls_loader = DataLoader(val_cls_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_cls_loader = DataLoader(test_cls_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Extracting train embeddings...")
train_embeddings, train_labels = extract_embeddings(encoder, train_cls_loader, device)
print("Extracting val embeddings...")
val_embeddings, val_labels = extract_embeddings(encoder, val_cls_loader, device)
print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(encoder, test_cls_loader, device)

print(f"\nTrain embeddings shape: {train_embeddings.shape}")
print(f"Val embeddings shape: {val_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")

In [ ]:
class EmbeddingDataset(Dataset):
    """Dataset for pre-extracted embeddings"""
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

train_emb_dataset = EmbeddingDataset(train_embeddings, train_labels)
val_emb_dataset = EmbeddingDataset(val_embeddings, val_labels)
test_emb_dataset = EmbeddingDataset(test_embeddings, test_labels)

train_emb_loader = DataLoader(train_emb_dataset, batch_size=64, shuffle=True)
val_emb_loader = DataLoader(val_emb_dataset, batch_size=64, shuffle=False)
test_emb_loader = DataLoader(test_emb_dataset, batch_size=64, shuffle=False)

In [ ]:
classifier = LinearClassifier(
    input_dim=encoder.hidden_size,
    num_classes=3,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
classifier_optimizer = torch.optim.Adam(classifier.parameters(), lr=CLASSIFIER_LR)

print(f"Classifier parameters: {sum(p.numel() for p in classifier.parameters()):,}")

In [ ]:
print("\nStage 2: Training Linear Classifier")
print("=" * 50)

best_val_f1 = 0.0
best_classifier_state = None

for epoch in range(CLASSIFIER_EPOCHS):
    classifier.train()
    epoch_loss = 0.0
    
    for embeddings, labels in train_emb_loader:
        embeddings = embeddings.to(device)
        labels = labels.to(device)
        
        logits = classifier(embeddings)
        loss = criterion(logits, labels)
        
        classifier_optimizer.zero_grad()
        loss.backward()
        classifier_optimizer.step()
        
        epoch_loss += loss.item()
    
    classifier.eval()
    val_preds = []
    val_true = []
    
    with torch.no_grad():
        for embeddings, labels in val_emb_loader:
            embeddings = embeddings.to(device)
            logits = classifier(embeddings)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average='macro')
    
    avg_loss = epoch_loss / len(train_emb_loader)
    print(f"Epoch {epoch+1}/{CLASSIFIER_EPOCHS} - Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_classifier_state = classifier.state_dict().copy()

print(f"\nBest validation F1: {best_val_f1:.4f}")
classifier.load_state_dict(best_classifier_state)

In [ ]:
print("\nFinal Evaluation")
print("=" * 50)

classifier.eval()
test_preds = []
test_true = []

with torch.no_grad():
    for embeddings, labels in test_emb_loader:
        embeddings = embeddings.to(device)
        logits = classifier(embeddings)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(labels.numpy())

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average='macro')

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

test_pred_labels = [id2label[p] for p in test_preds]
test_true_labels = [id2label[t] for t in test_true]

print("\nClassification Report:")
print(classification_report(test_true_labels, test_pred_labels, labels=clarity_labels, zero_division=0))

In [ ]:
import os
import zipfile

os.makedirs("./predictions", exist_ok=True)
os.makedirs("../predictions", exist_ok=True)

prediction_filename = "./predictions/prediction"
with open(prediction_filename, 'w') as f:
    for pred in test_pred_labels:
        f.write(f"{pred}\n")

zip_filename = "../predictions/prediction_contrastive.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prediction_filename, arcname="prediction")

print(f"Saved predictions to {zip_filename}")

In [ ]:
torch.save({
    'encoder_state_dict': encoder.state_dict(),
    'classifier_state_dict': classifier.state_dict(),
}, './contrastive_roberta_model.pt')

print("Model saved to ./contrastive_roberta_model.pt")